# 제11장: 알고리즘 감사와 인증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch11.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

알고리즘 감사는 AI 시스템이 설계 의도대로 작동하는지 **독립적으로 검증**하는 과정입니다.
감사 계획 수립, 상세 체크리스트(170개 항목), 기술 검증 프레임워크를 구현합니다.
이 장의 코드는 실제 감사 업무에 즉시 활용 가능합니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**AI Governance Audit Plan Generator**

### AI Governance Audit Plan Generator



In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import date, timedelta
from enum import Enum

class AuditType(Enum):
 """Audit Types"""
 SELF_ASSESSMENT = "Self-Assessment"
 INTERNAL = "Internal Audit"
 EXTERNAL = "External Audit"
 REGULATORY = "Regulatory Audit"

class RiskLevel(Enum):
 """Risk Levels"""
 HIGH = "High Risk"
 MEDIUM = "Medium Risk"
 LOW = "Low Risk"

@dataclass
class AuditScope:
 """Audit Scope"""
 systems: List[str] # Target System
 domains: List[str] # Audit 
 period: str # Target Period

@dataclass
class AuditPlan:
 """Audit """
 audit_id: str
 audit_name: str
 audit_type: AuditType
 
 # days 
 planning_start: date
 fieldwork_start: date
 fieldwork_end: date
 report_date: date
 
 # Scope
 scope: AuditScope
 
 # 
 lead_auditor: str
 audit_team: List[str]
 
 # 
 criteria: List[str]
 
 # Risk based Priority
 priority_areas: List[Dict[str, str]] = field(default_factory=list)


class AuditPlanGenerator:
 """Audit Generator"""
 
 # Standard Audit Domains
 STANDARD_DOMAINS = [
 "Policy & Strategy",
 "Organization & Roles",
 "Development Process",
 "Deployment & Approval",
 "Operation & Monitoring",
 "Fairness Management",
 "Explainability",
 "Human Oversight",
 "Documentation & Records",
 "User Rights",
 "Incident Management",
 ]
 
 # Standard Audit Criteria
 STANDARD_CRITERIA = [
 "AI Basic Act & Enforcement Decree",
 "Internal AI Ethics Policy",
 "Internal AI Governance Standards",
 "Industry-specific Regulations (if applicable)",
 "ISO/IEC 42001\cite{iso42001} (Reference)",
 ]
 
 def __init__(self, organization_name: str):
 self.organization = organization_name
 
 def generate_annual_plan(
 self,
 year: int,
 ai_systems: List[Dict],
 ) -> List[AuditPlan]:
 """Generate annual audit plan"""
 
 plans = []
 
 # Identify high-risk systems
 high_risk = [s for s in ai_systems if s.get('risk_level') == 'high']
 medium_risk = [s for s in ai_systems if s.get('risk_level') == 'medium']
 low_risk = [s for s in ai_systems if s.get('risk_level') == 'low']
 
 # Q1: High-risk system audit
 if high_risk:
 plans.append(AuditPlan(
 audit_id=f"AUD-{year}-001",
 audit_name=f"{year} High-Risk AI System Audit",
 audit_type=AuditType.INTERNAL,
 planning_start=date(year, 1, 15),
 fieldwork_start=date(year, 2, 1),
 fieldwork_end=date(year, 2, 28),
 report_date=date(year, 3, 15),
 scope=AuditScope(
 systems=[s['system_id'] for s in high_risk],
 domains=self.STANDARD_DOMAINS,
 period=f"Entire {year-1}",
 ),
 lead_auditor="TBD",
 audit_team=[],
 criteria=self.STANDARD_CRITERIA,
 priority_areas=[
 {"area": "Fairness Management", "reason": "Mandatory for high-risk"},
 {"area": "Human Oversight", "reason": "Regulatory requirement"},
 {"area": "Documentation & Records", "reason": "5-year retention obligation"},
 ],
 ))
 
 # Q2: Governance framework audit
 plans.append(AuditPlan(
 audit_id=f"AUD-{year}-002",
 audit_name=f"{year} AI Governance Framework Audit",
 audit_type=AuditType.INTERNAL,
 planning_start=date(year, 4, 1),
 fieldwork_start=date(year, 4, 15),
 fieldwork_end=date(year, 5, 15),
 report_date=date(year, 5, 31),
 scope=AuditScope(
 systems=["Enterprise"],
 domains=["Policy & Strategy", "Organization & Roles", "Incident Management"],
 period=f"July {year-1} - March {year}",
 ),
 lead_auditor="TBD",
 audit_team=[],
 criteria=self.STANDARD_CRITERIA[:3],
 priority_areas=[
 {"area": "Policy Update", "reason": "Reflect regulatory changes"},
 {"area": "Role Implementation", "reason": "Verify effectiveness"},
 ],
 ))
 
 # Q3: Medium-risk system sample audit
 if medium_risk:
 sample_size = min(5, len(medium_risk))
 sample = medium_risk[:sample_size]
 
 plans.append(AuditPlan(
 audit_id=f"AUD-{year}-003",
 audit_name=f"{year} Medium-Risk AI System Sample Audit",
 audit_type=AuditType.INTERNAL,
 planning_start=date(year, 7, 1),
 fieldwork_start=date(year, 7, 15),
 fieldwork_end=date(year, 8, 15),
 report_date=date(year, 8, 31),
 scope=AuditScope(
 systems=[s['system_id'] for s in sample],
 domains=self.STANDARD_DOMAINS[:7],
 period=f"First half of {year}",
 ),
 lead_auditor="TBD",
 audit_team=[],
 criteria=self.STANDARD_CRITERIA[:3],
 priority_areas=[
 {"area": "Monitoring", "reason": "Drift management"},
 ],
 ))
 
 # Q4: Annual Comprehensive Assessment
 plans.append(AuditPlan(
 audit_id=f"AUD-{year}-004",
 audit_name=f"{year} AI Governance Comprehensive Assessment",
 audit_type=AuditType.INTERNAL,
 planning_start=date(year, 10, 1),
 fieldwork_start=date(year, 10, 15),
 fieldwork_end=date(year, 11, 15),
 report_date=date(year, 11, 30),
 scope=AuditScope(
 systems=["Enterprise"],
 domains=self.STANDARD_DOMAINS,
 period=f"Entire {year}",
 ),
 lead_auditor="TBD",
 audit_team=[],
 criteria=self.STANDARD_CRITERIA,
 priority_areas=[
 {"area": "Maturity Assessment", "reason": "Measure annual progress"},
 {"area": "Remediation", "reason": "Follow-up on previous findings"},
 ],
 ))
 
 return plans
 
 def generate_plan_document(self, plan: AuditPlan) -> str:
 """Generate audit plan document"""
 
 doc = f"""
# AI Governance Audit Plan

---

## 1. Audit 

| Item | Content |
|------|------|
| Audit ID | {plan.audit_id} |
| Audit Name | {plan.audit_name} |
| Audit Type | {plan.audit_type.value} |
| Lead Auditor | {plan.lead_auditor} |

---

## 2. Audit days 

| Phase | Start | End |
|------|------|------|
| Planning | {plan.planning_start} | {plan.fieldwork_start - timedelta(days=1)} |
| Fieldwork | {plan.fieldwork_start} | {plan.fieldwork_end} |
| Reporting | {plan.fieldwork_end + timedelta(days=1)} | {plan.report_date} |

---

## 3. Audit Scope

### 3.1 Target System

"""
 
 for system in plan.scope.systems:
 doc += f"- {system}\n"
 
 doc += f"""

### 3.2 Audit 

"""
 
 for domain in plan.scope.domains:
 doc += f"- {domain}\n"
 
 doc += f"""

### 3.3 Target Period

{plan.scope.period}

---

## 4. Audit 

"""
 
 for i, criterion in enumerate(plan.criteria, 1):
 doc += f"{i}. {criterion}\n"
 
 doc += f"""

---

## 5. Risk based Priority

| | Priority |
|------|--------------|
"""
 
 for area in plan.priority_areas:
 doc += f"| {area['area']} | {area['reason']} |\n"
 
 doc += f"""

---

## 6. Audit 

| Role | |
|------|--------|
| Audit | {plan.lead_auditor} |
"""
 
 for i, member in enumerate(plan.audit_team, 1):
 doc += f"| Audit {i} | {member} |\n"
 
 doc += """

---

## 7. 

1. Audit Report
2. 3. 4. ---

## 8. Approval

| Role | Name | Signature | Date |
|------|------|------|------|
| Lead Auditor | | | |
| Head of Audit | | | |
| Dept. Head | | | |

---

*This audit plan is subject to change.*
"""
 
 return doc

**AI Governance Audit Checklist**

### AI Governance Audit Checklist

Detailed Check Items by Area

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum

class ComplianceStatus(Enum):
 """Compliance Status"""
 COMPLIANT = "Compliant"
 PARTIALLY_COMPLIANT = "Partially Compliant"
 NON_COMPLIANT = "Non-Compliant"
 NOT_APPLICABLE = "N/A"
 NOT_TESTED = "Not Tested"

@dataclass
class AuditCheckItem:
 """Audit Check Items"""
 item_id: str
 domain: str
 category: str
 question: str
 criteria: str
 
 # Check 
 status: ComplianceStatus = ComplianceStatus.NOT_TESTED
 evidence: str = ""
 finding: str = ""
 recommendation: str = ""


class AIGovernanceAuditChecklist:
 """AI Governance Audit Checklist"""
 
 def __init__(self):
 self.items: List[AuditCheckItem] = self._initialize_checklist()
 
 def _initialize_checklist(self) -> List[AuditCheckItem]:
 """ Checklist """
 
 items = []
 
 # 1. Policy & Strategy
 items.extend([
 AuditCheckItem(
 item_id="POL-01",
 domain="Policy & Strategy",
 category="Policy Existence",
 question="Is there a formal AI Ethics Policy?",
 criteria="Documented policy, executive approval",
 ),
 AuditCheckItem(
 item_id="POL-02",
 domain="Policy & Strategy",
 category="Policy Maintenance",
 question="Has the policy been reviewed/updated within the last year?",
 criteria="Annual review cycle",
 ),
 AuditCheckItem(
 item_id="POL-03",
 domain="Policy & Strategy",
 category="Policy Dissemination",
 question="Has the policy been disseminated to relevant staff?",
 criteria="Training records, awareness check",
 ),
 AuditCheckItem(
 item_id="POL-04",
 domain="Policy & Strategy",
 category="Regulatory Alignment",
 question="Does the policy reflect the latest regulations?",
 criteria="Tracking and reflecting legal changes",
 ),
 ])
 
 # 2. Role
 items.extend([
 AuditCheckItem(
 item_id="ORG-01",
 domain=" Role",
 category=" ",
 question="AI Governance / ?",
 criteria=" , ",
 ),
 AuditCheckItem(
 item_id="ORG-02",
 domain=" Role",
 category="Role Definition",
 question="RACI Role Definition ?",
 criteria="RACI Document, ",
 ),
 AuditCheckItem(
 item_id="ORG-03",
 domain=" Role",
 category="Resource ",
 question="Governance ?",
 criteria=" Analysis",
 ),
 AuditCheckItem(
 item_id="ORG-04",
 domain=" Role",
 category=" ",
 question=" ( , ) ?",
 criteria=" , ",
 ),
 ])
 
 # 3. AI System Management
 items.extend([
 AuditCheckItem(
 item_id="SYS-01",
 domain="AI System Management",
 category=" ",
 question=" AI System ?",
 criteria=" ",
 ),
 AuditCheckItem(
 item_id="SYS-02",
 domain="AI System Management",
 category=" Classification",
 question=" System Level Classification ?",
 criteria=" ",
 ),
 AuditCheckItem(
 item_id="SYS-03",
 domain="AI System Management",
 category="Model ",
 question=" System Model ?",
 criteria="Model ",
 ),
 ])
 
 # 4. items.extend([
 AuditCheckItem(
 item_id="DEV-01",
 domain=" ",
 category="Approval Gate",
 question=" Approval Operation ?",
 criteria="Approval ",
 ),
 AuditCheckItem(
 item_id="DEV-02",
 domain=" ",
 category="Approval Compliance",
 question=" Approval ?",
 criteria="Approval 0 ",
 ),
 AuditCheckItem(
 item_id="DEV-03",
 domain=" ",
 category="Data ",
 question="Learning Data Management ?",
 criteria="Data Check ",
 ),
 ])
 
 # 5. Fairness
 items.extend([
 AuditCheckItem(
 item_id="FAI-01",
 domain="Fairness",
 category="Assessment ",
 question=" System Fairness Assessment ?",
 criteria="Fairness Assessment Report",
 ),
 AuditCheckItem(
 item_id="FAI-02",
 domain="Fairness",
 category=" ",
 question="4/5 (DP Ratio >= 0.8) ?",
 criteria=" ",
 ),
 AuditCheckItem(
 item_id="FAI-03",
 domain="Fairness",
 category=" ",
 question=" Model ?",
 criteria=" ",
 ),
 AuditCheckItem(
 item_id="FAI-04",
 domain="Fairness",
 category="Monitoring",
 question="Fairness Monitoring ?",
 criteria="Monitoring , ",
 ),
 ])
 
 # 6. 
 items.extend([
 AuditCheckItem(
 item_id="XAI-01",
 domain=" ",
 category="Implementation",
 question=" System XAI Implementation ?",
 criteria="SHAP/LIME Implementation ",
 ),
 AuditCheckItem(
 item_id="XAI-02",
 domain=" ",
 category=" ",
 question=" ?",
 criteria=" ",
 ),
 AuditCheckItem(
 item_id="XAI-03",
 domain=" ",
 category=" ",
 question=" ?",
 criteria=" , ",
 ),
 ])
 
 # 7. items.extend([
 AuditCheckItem(
 item_id="HUM-01",
 domain=" ",
 category="Level Definition",
 question=" Level(HITL/HOTL/HIC) Definition ?",
 criteria=" , System Definition",
 ),
 AuditCheckItem(
 item_id="HUM-02",
 domain=" ",
 category=" ",
 question="Definition Level ?",
 criteria=" , ",
 ),
 AuditCheckItem(
 item_id="HUM-03",
 domain=" ",
 category=" ",
 question="AI ?",
 criteria=" ",
 ),
 ])
 
 # 8. Document items.extend([
 AuditCheckItem(
 item_id="DOC-01",
 domain="Document ",
 category=" ",
 question=" Document ?",
 criteria="Document Checklist",
 ),
 AuditCheckItem(
 item_id="DOC-02",
 domain="Document ",
 category=" ",
 question=" 5years Management ?",
 criteria=" System, ",
 ),
 AuditCheckItem(
 item_id="DOC-03",
 domain="Document ",
 category=" ",
 question=" / ?",
 criteria=" ",
 ),
 ])
 
 # 9. items.extend([
 AuditCheckItem(
 item_id="USR-01",
 domain=" ",
 category=" ",
 question="AI ?",
 criteria=" , ",
 ),
 AuditCheckItem(
 item_id="USR-02",
 domain=" ",
 category=" ",
 question=" ?",
 criteria=" , ",
 ),
 AuditCheckItem(
 item_id="USR-03",
 domain=" ",
 category=" ",
 question=" ?",
 criteria=" , Period",
 ),
 ])
 
 # 10. Management
 items.extend([
 AuditCheckItem(
 item_id="INC-01",
 domain=" Management",
 category=" ",
 question=" Response ?",
 criteria="Response Document",
 ),
 AuditCheckItem(
 item_id="INC-02",
 domain=" Management",
 category=" ",
 question=" .Management ?",
 criteria=" ",
 ),
 AuditCheckItem(
 item_id="INC-03",
 domain=" Management",
 category=" ",
 question=" ?",
 criteria=" ",
 ),
 ])
 
 return items
 
 def evaluate_item(
 self,
 item_id: str,
 status: ComplianceStatus,
 evidence: str,
 finding: str = "",
 recommendation: str = "",
 ) -> None:
 """Items Assessment"""
 
 for item in self.items:
 if item.item_id == item_id:
 item.status = status
 item.evidence = evidence
 item.finding = finding
 item.recommendation = recommendation
 break
 
 def get_summary(self) -> Dict:
 """ """
 
 total = len(self.items)
 status_counts = {}
 
 for item in self.items:
 status_counts[item.status.value] = status_counts.get(item.status.value, 0) + 1
 
 # Compliance Calculation ( None, Check )
 applicable = total - status_counts.get(" None", 0) - status_counts.get(" Check", 0)
 compliant = status_counts.get("Compliance", 0)
 
 compliance_rate = compliant / applicable if applicable > 0 else 0
 
 # Analysis
 domain_results = {}
 for item in self.items:
 if item.domain not in domain_results:
 domain_results[item.domain] = {'total': 0, 'compliant': 0}
 domain_results[item.domain]['total'] += 1
 if item.status == ComplianceStatus.COMPLIANT:
 domain_results[item.domain]['compliant'] += 1
 
 return {
 'total_items': total,
 'status_counts': status_counts,
 'compliance_rate': compliance_rate,
 'domain_results': domain_results,
 'findings_count': len([i for i in self.items if i.finding]),
 }
 
 def get_findings(self) -> List[Dict]:
 """ """
 
 findings = []
 
 for item in self.items:
 if item.status in [ComplianceStatus.NON_COMPLIANT, ComplianceStatus.PARTIALLY_COMPLIANT]:
 findings.append({
 'item_id': item.item_id,
 'domain': item.domain,
 'question': item.question,
 'status': item.status.value,
 'finding': item.finding,
 'recommendation': item.recommendation,
 })
 
 return findings
 
 def generate_report(self, audit_info: Dict) -> str:
 """Audit Report Generation"""
 
 summary = self.get_summary()
 findings = self.get_findings()
 
 report = f"""
# AI Governance Audit Report

---

## 1. Audit 

| Items | |
|------|------|
| Audit ID | {audit_info.get('audit_id', '')} |
| Audit | {audit_info.get('audit_name', '')} |
| Audit Period | {audit_info.get('period', '')} |
| Audit | {audit_info.get('lead_auditor', '')} |
| Report days | {audit_info.get('report_date', '')} |

---

## 2. ### AI Governance Compliance {summary['compliance_rate']:.1%} .

| Status | Items |
|------|--------|
"""
 
 for status, count in summary['status_counts'].items():
 report += f"| {status} | {count} |\n"
 
 report += f"""

### Major {len(findings)} .

"""
 
 # Compliance 
 non_compliant = [f for f in findings if f['status'] == ' Compliance']
 partial = [f for f in findings if f['status'] == ' Compliance']
 
 if non_compliant:
 report += f"- Compliance: {len(non_compliant)} ( )\n"
 if partial:
 report += f"- Compliance: {len(partial)} ( )\n"
 
 report += f"""

---

## 3. | | Check Items | Compliance | Compliance |
|------|----------|------|--------|
"""
 
 for domain, result in summary['domain_results'].items():
 rate = result['compliant'] / result['total'] if result['total'] > 0 else 0
 report += f"| {domain} | {result['total']} | {result['compliant']} | {rate:.0%} |\n"
 
 report += f"""

---

## 4. """
 
 for i, finding in enumerate(findings, 1):
 severity = "" if finding['status'] == ' Compliance' else ""
 report += f"""
### 4.{i} [{finding['item_id']}] {finding['domain']}

Check Items: {finding['question']}

Status: {severity} {finding['status']}

 : 
{finding['finding']}

 :
{finding['recommendation']}

---
"""
 
 report += f"""

## 5. ### ( Compliance)

"""
 
 for finding in non_compliant:
 report += f"- [{finding['item_id']}] {finding['recommendation']}\n"
 
 report += f"""

### ( Compliance)

"""
 
 for finding in partial:
 report += f"- [{finding['item_id']}] {finding['recommendation']}\n"
 
 report += f"""

---

## 6. | | ID | | | days |
|------|------------|----------|----------|----------|
"""
 
 for i, finding in enumerate(findings, 1):
 report += f"| {i} | {finding['item_id']} | | | |\n"
 
 report += """

---

## 7. 

1. Audit Checklist 
2. 3. ---

 : Audit 
 : Audit 
Approval: Audit 

* Report 15 5years .*
"""
 
 return report

**Algorithmic Audit Framework**

### Algorithmic Audit Framework

Technical Validation for Fairness, Bias, and Transparency

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from datetime import datetime

@dataclass
class AlgorithmicAuditConfig:
 """Algorithm Audit Configuration"""
 
 # Fairness threshold
 dp_ratio_threshold: float = 0.8
 eo_diff_threshold: float = 0.1
 
 # Sample size
 min_sample_size: int = 1000
 min_subgroup_size: int = 100
 
 # Confidence interval
 confidence_level: float = 0.95


@dataclass
class AuditResult:
 """Audit Result"""
 metric_name: str
 value: float
 threshold: float
 passed: bool
 confidence_interval: Tuple[float, float]
 sample_size: int
 details: Dict = field(default_factory=dict)


class AlgorithmicAuditor:
 """
 Algorithmic Auditor
 Conducts independent technical audits
 """
 
 def __init__(self, config: Optional[AlgorithmicAuditConfig] = None):
 self.config = config or AlgorithmicAuditConfig()
 self.results: List[AuditResult] = []
 
 def audit_fairness(
 self,
 y_true: np.ndarray,
 y_pred: np.ndarray,
 sensitive_features: pd.Series,
 ) -> List[AuditResult]:
 """Fairness Audit"""
 
 results = []
 
 # 1. Demographic Parity
 groups = sensitive_features.unique()
 selection_rates = {}
 
 for group in groups:
 mask = sensitive_features == group
 n = mask.sum()
 if n >= self.config.min_subgroup_size:
 rate = y_pred[mask].mean()
 selection_rates[group] = {'rate': rate, 'n': n}
 
 if len(selection_rates) >= 2:
 rates = [v['rate'] for v in selection_rates.values()]
 dp_ratio = min(rates) / max(rates) if max(rates) > 0 else 1.0
 
 # Calculate confidence interval (bootstrap)
 ci = self._bootstrap_ci(
 y_pred, sensitive_features,
 lambda yp, sf: self._calculate_dp_ratio(yp, sf)
 )
 
 results.append(AuditResult(
 metric_name="Demographic Parity Ratio",
 value=dp_ratio,
 threshold=self.config.dp_ratio_threshold,
 passed=dp_ratio >= self.config.dp_ratio_threshold,
 confidence_interval=ci,
 sample_size=len(y_pred),
 details={'group_rates': selection_rates},
 ))
 
 # 2. Equalized Odds
 tpr_by_group = {}
 fpr_by_group = {}
 
 for group in groups:
 mask = sensitive_features == group
 y_t = y_true[mask]
 y_p = y_pred[mask]
 
 # TPR
 pos_mask = y_t == 1
 if pos_mask.sum() >= self.config.min_subgroup_size:
 tpr = y_p[pos_mask].mean()
 tpr_by_group[group] = tpr
 
 # FPR
 neg_mask = y_t == 0
 if neg_mask.sum() >= self.config.min_subgroup_size:
 fpr = y_p[neg_mask].mean()
 fpr_by_group[group] = fpr
 
 if len(tpr_by_group) >= 2:
 tpr_diff = max(tpr_by_group.values()) - min(tpr_by_group.values())
 fpr_diff = max(fpr_by_group.values()) - min(fpr_by_group.values())
 eo_diff = max(tpr_diff, fpr_diff)
 
 results.append(AuditResult(
 metric_name="Equalized Odds Difference",
 value=eo_diff,
 threshold=self.config.eo_diff_threshold,
 passed=eo_diff <= self.config.eo_diff_threshold,
 confidence_interval=(0, 0), # Simplified
 sample_size=len(y_pred),
 details={'tpr_by_group': tpr_by_group, 'fpr_by_group': fpr_by_group},
 ))
 
 self.results.extend(results)
 return results
 
 def _calculate_dp_ratio(self, y_pred, sensitive_features):
 """Calculate DP Ratio"""
 groups = sensitive_features.unique()
 rates = []
 for group in groups:
 mask = sensitive_features == group
 if mask.sum() >= 30:
 rates.append(y_pred[mask].mean())
 return min(rates) / max(rates) if rates and max(rates) > 0 else 1.0
 
 def _bootstrap_ci(self, y_pred, sensitive_features, metric_fn, n_bootstrap=1000):
 """Bootstrap Confidence Interval"""
 
 n = len(y_pred)
 bootstrap_values = []
 
 for _ in range(n_bootstrap):
 indices = np.random.choice(n, size=n, replace=True)
 value = metric_fn(y_pred[indices], sensitive_features.iloc[indices])
 bootstrap_values.append(value)
 
 alpha = 1 - self.config.confidence_level
 lower = np.percentile(bootstrap_values, alpha/2 * 100)
 upper = np.percentile(bootstrap_values, (1 - alpha/2) * 100)
 
 return (lower, upper)
 
 def audit_data_representativeness(
 self,
 training_data: pd.DataFrame,
 production_data: pd.DataFrame,
 sensitive_column: str,
 ) -> AuditResult:
 """Data Representativeness Audit"""
 
 train_dist = training_data[sensitive_column].value_counts(normalize=True)
 prod_dist = production_data[sensitive_column].value_counts(normalize=True)
 
 # Calculate distribution difference (Total Variation Distance)
 all_groups = set(train_dist.index) | set(prod_dist.index)
 tvd = 0.5 * sum(
 abs(train_dist.get(g, 0) - prod_dist.get(g, 0))
 for g in all_groups
 )
 
 result = AuditResult(
 metric_name="Data Representativeness (TVD)",
 value=tvd,
 threshold=0.1, # 10% difference allowed
 passed=tvd <= 0.1,
 confidence_interval=(0, 0),
 sample_size=len(training_data) + len(production_data),
 details={
 'training_distribution': train_dist.to_dict(),
 'production_distribution': prod_dist.to_dict(),
 },
 )
 
 self.results.append(result)
 return result
 
 def audit_model_stability(
 self,
 predictions_t1: np.ndarray,
 predictions_t2: np.ndarray,
 ) -> AuditResult:
 """Model Stability Audit"""
 
 # Prediction change rate
 changes = predictions_t1 != predictions_t2
 change_rate = changes.mean()
 
 result = AuditResult(
 metric_name="Prediction Stability",
 value=1 - change_rate,
 threshold=0.95, # 95%+ consistency
 passed=(1 - change_rate) >= 0.95,
 confidence_interval=(0, 0),
 sample_size=len(predictions_t1),
 details={'change_rate': change_rate},
 )
 
 self.results.append(result)
 return result
 
 def generate_audit_report(
 self,
 system_info: Dict,
 auditor_info: Dict,
 ) -> str:
 """Audit Report Generation"""
 
 passed_count = sum(1 for r in self.results if r.passed)
 total_count = len(self.results)
 
 report = f"""
# Algorithm Audit Report

---

## 1. Audit Overview

### 1.1 Target System

| Item | Content |
|------|------|
| System Name | {system_info.get('system_name', '')} |
| System ID | {system_info.get('system_id', '')} |
| Version | {system_info.get('version', '')} |
| Purpose | {system_info.get('purpose', '')} |

### 1.2 Audit Information

| Item | Content |
|------|------|
| Organization | {auditor_info.get('organization', '')} |
| Auditor | {auditor_info.get('auditor_name', '')} |
| Date | {auditor_info.get('audit_date', '')} |
| Criteria | {auditor_info.get('criteria', '')} |

---

## 2. ### Summary Results

- Total Checkpoints: {total_count}
- Passed: {passed_count}
- Failed: {total_count - passed_count}
- Pass Rate: {passed_count/total_count*100 if total_count > 0 else 0:.1f}%

### """
 
 if passed_count == total_count:
 report += "[PASS] All fairness criteria met.\n"
 elif passed_count / total_count >= 0.8:
 report += "[WARN] Most criteria met, but some improvements required.\n"
 else:
 report += "[FAIL] Multiple fairness criteria failed. Immediate action required.\n"
 
 report += f"""

---

## 3. Detailed Results

| Metric | Measured Value | Threshold | Result | Conf. Interval |
|--------|--------|------|------|----------|
"""
 
 for result in self.results:
 status = "[PASS] Pass" if result.passed else "[FAIL] Fail"
 ci = f"[{result.confidence_interval[0]:.3f}, {result.confidence_interval[1]:.3f}]" if result.confidence_interval != (0, 0) else "-"
 
 threshold_str = f">= {result.threshold}" if "Ratio" in result.metric_name or "Stability" in result.metric_name else f"<= {result.threshold}"
 
 report += f"| {result.metric_name} | {result.value:.4f} | {threshold_str} | {status} | {ci} |\n"
 
 report += f"""

---

## 4. Detailed Analysis

"""
 
 for i, result in enumerate(self.results, 1):
 report += f"""
### 4.{i} {result.metric_name}

Measured Value: {result.value:.4f} 
Threshold: {'>=' if 'Ratio' in result.metric_name or 'Stability' in result.metric_name else '<='} {result.threshold} 
Result: {'Pass' if result.passed else 'Fail'} 
Sample Size: {result.sample_size:,}

"""
 
 if 'group_rates' in result.details:
 report += "Group Selection Rates:\n\n"
 report += "| Group | Rate | Sample Size |\n|------|--------|--------|\n"
 for group, stats in result.details['group_rates'].items():
 report += f"| {group} | {stats['rate']:.4f} | {stats['n']:,} |\n"
 report += "\n"
 
 report += f"""
---

## 5. Recommendations

"""
 
 failed_results = [r for r in self.results if not r.passed]
 
 if failed_results:
 for result in failed_results:
 report += f"- {result.metric_name}: Threshold not met. Re-training or bias mitigation recommended.\n"
 else:
 report += "- Currently within thresholds; continuous monitoring recommended.\n"
 
 report += f"""

---

## 6. Disclaimer

This audit was performed based on provided data and information.
The results reflect the state at the time of the audit and may vary
with subsequent system or data changes.

---

Organization: {auditor_info.get('organization', '')} 
Auditor: {auditor_info.get('auditor_name', '')} 
Date: {auditor_info.get('audit_date', '')}

*This report is retained for 5 years per Article 15.*
"""
 
 return report